## Connect to NYC Open Data

In [1]:

from pathlib import Path
import pandas as pd
import requests

# Official NYC collision dataset API
API_URL = "https://data.cityofnewyork.us/resource/h9gi-nx95.json"

# Initial date range
START_DATE = "2019-01-01"
END_DATE = "2026-01-01"  # Exclusive: includes 2025

# Columns needed for RoadRisk
COLUMNS = [
    "collision_id",
    "crash_date",
    "crash_time",
    "borough",
    "latitude",
    "longitude",
    "on_street_name"
]

# Test the API with just 5 records
params = {
    "$select": ", ".join(COLUMNS),
    "$where": (
        f"crash_date >= '{START_DATE}' AND "
        f"crash_date < '{END_DATE}' AND "
        "borough = 'MANHATTAN'"
    ),
    "$limit": 5
}

response = requests.get(API_URL, params=params, timeout=30)
response.raise_for_status()

sample = pd.DataFrame(response.json())
display(sample)

print("Records retrieved:", len(sample))
print("API connection successful!")


,collision_id,crash_date,crash_time,borough,latitude,longitude,on_street_name
0,8d539e78e873420d71e320416390af2c,2025-12-31T00:00:00.000,03:00,MANHATTAN,40.81022,-73.949455,NaN
1,9a84ab3011469a48c2833a35507520d0,2025-12-31T00:00:00.000,21:50,MANHATTAN,40.7237,-74.00791,HUDSON ST
2,5e161e788e11d939e8c92c8aab77edd1,2025-12-31T00:00:00.000,13:32,MANHATTAN,40.816692,-73.945206,NaN
3,1c27b3ac8d5e5f9efb45e5f5fc4ed4e5,2025-12-31T00:00:00.000,15:10,MANHATTAN,40.768364,-73.98264,NaN
4,8b4f71ead8b1ae2148f66bd19f7631e4,2025-12-31T00:00:00.000,17:25,MANHATTAN,40.758797,-73.97559,NaN


Records retrieved: 5
API connection successful!


In [9]:

OUTPUT = "../data/raw/manhattan_collisions.csv"
Path(OUTPUT).parent.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 10000
last_id = None
total_rows = 0

base_filter = (
    f"crash_date >= '{START_DATE}' AND "
    f"crash_date < '{END_DATE}' AND "
    "borough = 'MANHATTAN'"
)

while True:
    where = base_filter

    if last_id is not None:
        safe_id = last_id.replace("'", "''")
        where += f" AND collision_id > '{safe_id}'"

    params = {
        "$select": ", ".join(COLUMNS),
        "$where": where,
        "$order": "collision_id ASC",
        "$limit": BATCH_SIZE
    }

    response = requests.get(API_URL, params=params, timeout=120)

    if not response.ok:
        print("API error:", response.text[:1000])
        response.raise_for_status()

    records = response.json()

    if not records:
        break

    batch = pd.DataFrame(records).reindex(columns=COLUMNS)

    batch.to_csv(
        OUTPUT,
        mode="w" if total_rows == 0 else "a",
        header=(total_rows == 0),
        index=False
    )

    total_rows += len(batch)
    last_id = str(batch["collision_id"].iloc[-1])

    print(f"Downloaded {total_rows:,} records")

    if len(records) < BATCH_SIZE:
        break

print(f"\nDownload complete: {total_rows:,} records")
print(f"Saved to: {OUTPUT}")


Downloaded 10,000 records
Downloaded 20,000 records
Downloaded 30,000 records
Downloaded 40,000 records
Downloaded 50,000 records
Downloaded 60,000 records
Downloaded 70,000 records
Downloaded 80,000 records
Downloaded 90,000 records
Downloaded 99,101 records

Download complete: 99,101 records
Saved to: ../data/raw/manhattan_collisions.csv


In [10]:

# Cell 3: Verify downloaded dataset

df = pd.read_csv(OUTPUT)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Duplicate IDs:", df["collision_id"].duplicated().sum())
print("Missing coordinates:", df["latitude"].isna().sum())

display(df.head())

assert len(df) == total_rows
assert df["collision_id"].is_unique
assert df["borough"].eq("MANHATTAN").all()

print("Download verification passed!")


Dataset shape: (99101, 7)
Columns: ['collision_id', 'crash_date', 'crash_time', 'borough', 'latitude', 'longitude', 'on_street_name']
Duplicate IDs: 0
Missing coordinates: 2734


,collision_id,crash_date,crash_time,borough,latitude,longitude,on_street_name
0,000008347c2c9ef23a99545a0a815898,2023-08-25T00:00:00.000,19:56,MANHATTAN,40.709354,-74.00816,NaN
1,0000a5c3f314ea75418497e73c018fe1,2020-01-18T00:00:00.000,13:30,MANHATTAN,40.753242,-73.96662,1 AVENUE
2,0001ac4d9de42ec118e73c0358c0487b,2024-04-27T00:00:00.000,22:07,MANHATTAN,40.784590,-73.94198,EAST 99 STREET
3,0001ffa7292a1d4d324b9d74ee7c2d25,2023-04-04T00:00:00.000,23:00,MANHATTAN,40.724003,-73.99102,NaN
4,0002965ce22b53380166aad83371220a,2020-07-11T00:00:00.000,17:11,MANHATTAN,40.779602,-73.95553,NaN


Download verification passed!


In [11]:

# Cell 4: Check completeness and year-wise coverage

# Ask the API how many records match our filter
response = requests.get(
    API_URL,
    params={
        "$select": "count(*) as total",
        "$where": base_filter
    },
    timeout=30
)
response.raise_for_status()

api_total = int(response.json()[0]["total"])
local_total = len(df)

print("API matching records:", api_total)
print("Downloaded records:", local_total)
print("Difference:", api_total - local_total)

# Check collision ID format
print("\nSample collision IDs:")
print(df["collision_id"].head().tolist())

# Examine annual counts
df["crash_date"] = pd.to_datetime(
    df["crash_date"], errors="coerce"
)

print("\nCollisions by year:")
print(
    df.groupby(df["crash_date"].dt.year)
      .size()
      .to_string()
)

print("\nEarliest date:", df["crash_date"].min())
print("Latest date:", df["crash_date"].max())
print("Missing dates:", df["crash_date"].isna().sum())

print("\nDownload complete:", api_total == local_total)


API matching records: 99101
Downloaded records: 99101
Difference: 0

Sample collision IDs:
['000008347c2c9ef23a99545a0a815898', '0000a5c3f314ea75418497e73c018fe1', '0001ac4d9de42ec118e73c0358c0487b', '0001ffa7292a1d4d324b9d74ee7c2d25', '0002965ce22b53380166aad83371220a']

Collisions by year:
crash_date
2019    26594
2020    10857
2021    11434
2022    11991
2023    11826
2024    11911
2025    14488

Earliest date: 2019-01-01 00:00:00
Latest date: 2025-12-31 00:00:00
Missing dates: 0

Download complete: True


In [14]:

# Cell 5: Initial data-quality analysis

# Preserve the original data
audit = df.copy()

audit["crash_date"] = pd.to_datetime(
    audit["crash_date"], errors="coerce"
)

audit["latitude"] = pd.to_numeric(
    audit["latitude"], errors="coerce"
)
audit["longitude"] = pd.to_numeric(
    audit["longitude"], errors="coerce"
)

# Identify missing coordinates
missing_coords = (
    audit[["latitude", "longitude"]]
    .isna()
    .any(axis=1)
)

# Check coordinates against broad NYC bounds
valid_coords = (
    audit["latitude"].between(40.5, 41.0)
    & audit["longitude"].between(-74.3, -73.6)
)

print("Total records:", len(audit))
print("Missing dates:", audit["crash_date"].isna().sum())
print("Duplicate IDs:", audit["collision_id"].duplicated().sum())

print("\nMissing coordinates:", missing_coords.sum())
print("Missing %:", round(missing_coords.mean() * 100, 2))

print(
    "Invalid coordinates (excluding missing):",
    (~valid_coords & ~missing_coords).sum()
)

print("\nCollisions by year:")
display(
    audit.groupby(audit["crash_date"].dt.year)
         .size()
         .rename("collisions")
         .to_frame()
)

print("\nCollisions by quarter:")
display(
    audit.groupby(audit["crash_date"].dt.to_period("Q"))
         .size()
         .rename("collisions")
         .to_frame()
)


Total records: 99101
Missing dates: 0
Duplicate IDs: 0

Missing coordinates: 2734
Missing %: 2.76
Invalid coordinates (excluding missing): 1029

Collisions by year:


,collisions
crash_date,
2019,26594
2020,10857
2021,11434
2022,11991
2023,11826
2024,11911
2025,14488



Collisions by quarter:


,collisions
crash_date,
2019Q1,6358
2019Q2,7063
2019Q3,6803
2019Q4,6370
2020Q1,4392
2020Q2,1420
2020Q3,2560
2020Q4,2485
2021Q1,2032
